# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and process the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcommons.org/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install -q mlcroissant matplotlib

## 1. Data Loading
Load the dataset metadata and record sets from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
from collections import namedtuple

# Croissant schema URL for FAIR^2
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from Croissant
dataset = mlc.Dataset(croissant_url)
# Metadata is available as a DatasetMetadata object
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Let's inspect the available record sets and their fields. We will use the `@id` field to reference record sets and fields consistently across the notebook. The following displays the available record sets, their IDs, and their available fields (with `@id` and name):

In [ ]:
# List all available record sets in the dataset, along with their fields and columns.

def describe_record_sets(ds):
    record_sets = ds.recordsets
    if not record_sets:
        print("No record sets discovered in metadata.")
        return []
    print(f"Discovered {len(record_sets)} record set(s):\n")
    record_set_ids = []
    for rs in record_sets:
        print(f"  Record Set Name: {rs.name}\n    @id: {rs.id}")
        record_set_ids.append(rs.id)
        # Print field summaries with @id
        print("    Fields:")
        for field in rs.fields:
            print(f"      - Name: {field.name}, @id: {field.id}, dataType: {getattr(field, 'dataType', None)}")
        print()
    return record_set_ids

record_set_ids = describe_record_sets(dataset)

if not record_set_ids:
    print("No record sets found. Please ensure your Croissant schema provides record sets.")
else:
    # As an example: print the first record of the first record set
    print("Sample record from the first record set:")
    sample_rs_id = record_set_ids[0]
    for rec in dataset.records(record_set=sample_rs_id):
        pprint.pprint(rec)
        break

## 3. Data Extraction
Let's load the full data for each available record set into a DataFrame. We'll use the record set `@id` identifiers (as listed above). If the dataset has multiple record sets, they will all be loaded for flexible exploration.

In [ ]:
# Extract all discovered record sets into pandas DataFrames, using their `@id`s as dictionary keys
dataframes = dict()
print("Loading records from each record set:\n")
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  - {record_set_id}: {df.shape[0]} rows, {df.shape[1]} columns")

if record_set_ids:
    # Display available columns in the first record set
    current_rs = record_set_ids[0]
    print(f"\nColumns in first record set ('{current_rs}'):")
    print(dataframes[current_rs].columns.tolist())
    print("\nFirst 5 records:")
    display(dataframes[current_rs].head())

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate simple analysis on one numeric field using field `@id` (e.g., filter records above a threshold, normalize, and group by a key). Adjust the field IDs according to those discovered above.

In [ ]:
# --- Adjust these variables to match your dataset schema ---
# For this example, we will select fields and record set IDs dynamically from the discovered metadata.
if not record_set_ids:
    print("No record sets found to perform analysis.")
else:
    # Select the first record set for demonstration
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]

    # Attempt to pick a numeric field for demonstration (float or integer)
    recordset_obj = next(rs for rs in dataset.recordsets if rs.id == record_set_id)
    numeric_fields = [field for field in recordset_obj.fields if getattr(field, 'dataType', None) in ('Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer')]
    if not numeric_fields:
        print("No numeric fields available in record set for EDA. Please check dataset schema.")
    else:
        # Pick the first numeric field's @id
        numeric_field = numeric_fields[0].id
        print(f"Using numeric field '{numeric_fields[0].name}' (@id: {numeric_field}) for analysis.\n")

        # Remove missing or non-numeric entries
        filtered_numeric = pd.to_numeric(df.get(numeric_field, pd.Series()), errors='coerce')
        df = df.assign(**{numeric_field: filtered_numeric})

        # Set an arbitrary threshold (10th percentile)
        threshold = df[numeric_field].quantile(0.9) if df[numeric_field].notnull().any() else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalize this numeric field among the filtered records
        if filtered_df[numeric_field].notnull().any():
            filtered_df[f"{numeric_field}_normalized"] = (
                filtered_df[numeric_field] - filtered_df[numeric_field].mean()
            ) / filtered_df[numeric_field].std()
            print(f"Normalized {numeric_field} for filtered records:")
            display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Attempt to group by a non-numeric field
        group_fields = [fld for fld in recordset_obj.fields if getattr(fld, 'dataType', None) not in ('Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer')]
        if group_fields:
            group_field = group_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
                print(f"Grouped data by '{group_fields[0].name}' (@id: {group_field}):")
                display(grouped_df.head())
            else:
                print(f"Grouping field '{group_fields[0].name}' (@id: {group_field}) not found in columns.")
        else:
            print("No suitable field to group by was found.")

## 5. Visualization
Let's visualize data distributions for the selected numeric field and show group-wise means, if grouping was performed.

In [ ]:
import matplotlib.pyplot as plt

if not record_set_ids or not numeric_fields:
    print("No data or numeric fields for visualization.")
else:
    # Histogram of the numeric field
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=20)
    plt.xlabel(numeric_fields[0].name)
    plt.ylabel('Count')
    plt.title(f"Distribution of {numeric_fields[0].name} [{numeric_field}]")
    plt.show()

    # Grouped means bar plot
    if group_fields and group_field in filtered_df.columns:
        means = filtered_df.groupby(group_field)[numeric_field].mean().dropna()
        if not means.empty:
            means.plot(kind='bar', figsize=(8,4), ylabel=numeric_fields[0].name)
            plt.title(f"Mean {numeric_fields[0].name} by {group_fields[0].name}")
            plt.xlabel(group_fields[0].name)
            plt.show()

## 6. Conclusion
In this notebook, we have:
- Loaded the FAIR^2 dataset metadata and explored its record sets using `mlcroissant`
- Extracted full records using the record set and field `@id` values
- Performed a simple EDA including filtering, normalization, and grouping on a numeric field
- Visualized basic data distributions

This workflow may be easily customized for other Croissant datasets and for more in-depth domain-specific analyses.